In [0]:
from pyspark.sql.types import StructType, StructField, StringType, VariantType, LongType, BooleanType, ArrayType

schema = StructType([
    StructField("id", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("actor", StructType([
        StructField("avatar_url", StringType(), True),
        StructField("display_login", StringType(), True),
        StructField("gravatar_id", StringType(), True),
        StructField("id", LongType(), True),
        StructField("login", StringType(), True),
        StructField("url", StringType(), True)
    ]), True),
    StructField("org", StructType([
        StructField("avatar_url", StringType(), True),
        StructField("gravatar_id", StringType(), True),
        StructField("id", LongType(), True),
        StructField("login", StringType(), True),
        StructField("url", StringType(), True)
    ]), True),
    StructField("payload", VariantType(), True),
    StructField("public", BooleanType(), True),
    StructField("repo", StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("url", StringType(), True)
    ]), True),
    StructField("type", StringType(), True)
])

In [0]:
RAW_DATA_PATH = "/Volumes/gh_archive_volume/raw/gh_archive/*/*/*/*.json.gz"
BRONZE_TABLE_TARGET = "main.default.raw_gh_archive"
CHECKPOINT_PATH = "/Volumes/gh_archive_lakehouse_dev/default/gh_archive_volume/_checkpoints/bronze_ingestion"

incremental_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema)
    .load(RAW_DATA_PATH)
)


In [0]:
query = (
    incremental_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(once=True) 
    .toTable(BRONZE_TABLE_TARGET)
)